# 54. Does the generator condition on the task code?

## Why this exists

Notebook 48, test 4, froze the generator and trained only a fresh 32-number
code. Every reading landed between 4.14 and 5.84 percent against 5 percent
chance, at boundary 0 and at boundary 9. So a code on its own cannot steer this
generator to a new group of twenty classes.

That answers one direction. It leaves the other direction open, and nothing in
the study has measured it: once the generator has been trained on a group, does
the code that group was trained with still matter?

The two questions are not the same. Test 4 asked whether a code can rescue a
generator that never saw the group. This asks whether the generator still needs
the code for a group it has already learned.

## The test

Take a saved model. For every group it learned, generate the target weights
three times and score the same test pictures with each.

| arm | the code used |
| --- | --- |
| `trained` | the code this group was trained with |
| `initial` | the code this group started from, before any training |
| `other` | a different learned group's trained code |

Nothing is trained here. Three sets of weights are generated and three
evaluations are run, so one saved model takes a few minutes.

## What each outcome means

The arms are read against 5 percent chance, which is one class in twenty.

- `initial` near chance and `other` near chance. The generator is conditioning
  on the code, as the design intends. The question closes.
- `initial` near `trained`. The generator writes a network that solves the
  group whatever code it is handed. The 32 numbers are then decoration for that
  group, and the paper has to say so.
- `initial` near `trained` but `other` near chance. The generator is
  conditioning on something coarse: enough to tell groups apart, not enough to
  need the trained code.

## The one thing that could spoil it

Each group's running BatchNorm statistics were collected while its trained code
was in use, so they fit the weights that code generates. Scoring a different
code through those same statistics compares two changes at once. Every arm here
therefore re-estimates its own statistics, from the same twenty training
batches of the same group, starting from the neutral template.

The `trained` arm is also scored a second time through the statistics the run
itself saved. If those two numbers disagree, the re-estimation is doing
something of its own and nothing below can be read.


## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules

# Importing uncle twice in one session gives two copies of the module with two
# copies of its constants, which is a confusing way to fail much later.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-indicators")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/sumitasthana/PATH.git", str(REPO)],
                       check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition_10x20.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this from a checkout that has the ten-group split.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None and IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package],
                       check=True)
sys.path.insert(0, str(REPO))

import torch
from torch.func import functional_call
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from uncle import checkpoint
from uncle.config import Config, random_stream, study_defaults
from uncle.hypernet import HyperNetwork, build_target
from uncle.measure import probe
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS
from uncle.trainer import UnCLe

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                               text=True).strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("Running at:", HEAD)
print("Device:", GPU_NAME)

## 2. Settings

In [ ]:
STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

# Which saved models to read. Boundary 9 is the end of the stream, where every
# group has been learned and the generator holds all nine codes, so one file
# gives nine readings. Adding boundary 4 would turn the answer from "the code
# does or does not matter at the end" into "and it does or does not change over
# the stream", at the cost of a second download per run.
#
# Each file is about 540 MB and is deleted as soon as it has been read.
ORDERS = ["order_A"]
SEEDS = [0, 1, 2, 3, 4]
POINTS = [9]

# How many training batches each arm uses to re-estimate its own BatchNorm
# statistics. Twenty batches of 64 is 1,280 pictures, which is plenty for a
# running mean and variance and costs a few seconds.
BUFFER_BATCHES = 20

# Held fixed, matching the streams these checkpoints came from.
BATCH_SIZE = 64
CLASSES_HINT = 20

RESULT_KEY = f"uncle/code_conditioning/{STUDY_ID}/arms_by_task.jsonl"

# The group each ordering unlearns at the end. Needed only to rebuild a run's
# configuration, because the request list is part of what names the run. No
# forgetting has happened at boundary 9: order_A is nine learn requests and the
# forget request comes after the last saved boundary.
FORGET = {"order_A": "0", "order_B": "4", "order_C": "8"}

DATA_ROOT = (Path("/content/data/tiny-imagenet-200") if IN_COLAB
             else REPO / "data/tiny-imagenet-200")
CACHE = (Path("/content/code-conditioning") if IN_COLAB
         else REPO / "outputs/code-conditioning")
CACHE.mkdir(parents=True, exist_ok=True)

RUNS = [(o, s) for o in ORDERS for s in SEEDS]
print(f"{len(RUNS)} runs x {len(POINTS)} boundaries = {len(RUNS) * len(POINTS)} "
      f"model files (about {len(RUNS) * len(POINTS) * 0.54:.0f} GB to download)")
print("Each file gives up to nine groups x three arms. Nothing is trained.")

## 3. Connect to R2

In [ ]:
def secret(name):
    """Read a credential from Colab secrets, or the environment elsewhere."""
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    value = os.environ.get(name)
    if value is None:
        env = REPO / ".env"
        if env.exists():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith(name + "="):
                    return line.partition("=")[2].strip().strip('"').strip("'")
    return value


store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))

# The settings the streams were trained with, read back rather than retyped, so
# the model this notebook rebuilds has the shape its checkpoint expects.
plan = json.loads(store.download(
    f"uncle/indicators/{STUDY_ID}/job2_summary.json", CACHE).read_text(encoding="utf-8"))
CLASSES = plan["classes_per_task"]
assert CLASSES == CLASSES_HINT, f"settings assume {CLASSES_HINT}, the runs used {CLASSES}"
CHANCE = 100.0 / CLASSES
print("Streams used learning rate", plan["learning_rate"], "| epochs", plan["epochs"],
      "| batch", plan["batch_size"], "| classes per group", CLASSES)
print(f"Chance accuracy is {CHANCE:.1f} percent.")

# Fail now rather than an hour in, if a model this session needs is missing.
missing = []
for order, seed in RUNS:
    for point in POINTS:
        key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
        if store.info(key) is None:
            missing.append(key)
if missing:
    raise RuntimeError("Missing models:\n" + "\n".join(sorted(set(missing))))
print("Every model this session needs is in R2.")

## 4. Build the pictures

Every group's own training and test splits, from the same partition and the
same partition seed the streams used. The training split is used here only to
re-estimate BatchNorm statistics, under `no_grad`. Nothing is fitted to it.

In [ ]:
STREAM_TASKS = [str(i) for i in range(9)]
tasks = build_tasks(root=DATA_ROOT, partition_path=PARTITIONS[20],
                    partition_seed=42, include=tuple(STREAM_TASKS), download=True)
print("Loaded Tiny ImageNet images for", len(STREAM_TASKS), "groups")
for name in STREAM_TASKS[:2]:
    print(f"  group {name}: {len(tasks[name]['train'])} train, "
          f"{len(tasks[name]['test'])} test")

## 5. Building a model at one boundary

This rebuilds the run as it was configured and loads the saved state into it.
`checkpoint.restore` recreates the task codes first, because they live in a
dictionary that starts empty and grows one entry per learned group. After
loading a boundary-9 model the generator holds all nine codes that stream
trained, and the trainer holds each group's saved BatchNorm statistics.

`starting_code` is what makes the `initial` arm possible at all. A group's
first code is drawn from a stream keyed by the run seed, the word "code" and
the group's name, and by nothing else, so it can be rebuilt exactly rather than
having had to be stored at the time.

In [ ]:
import gc
import random
import numpy as np


def build(order, seed, point):
    """The model this stream had at `point`, plus the config that names it."""
    settings = study_defaults(order)
    settings["requests"] = settings["requests"] + (("forget", FORGET[order]),)
    config = Config(**settings, seed=seed,
                    learning_rate=plan["learning_rate"], epochs=plan["epochs"],
                    batch_size=plan["batch_size"],
                    initialization=plan["initialization"], device=DEVICE)
    # The generator's starting weights come from the global random number
    # generator, not from config.seed by itself, so a rebuilt model only
    # matches the study's if these are set the same way. At a loaded boundary
    # the checkpoint overwrites all of it, so this is belt and braces here.
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(config.seed)
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
    path = store.download(key, CACHE)
    payload = torch.load(path, map_location=DEVICE, weights_only=False)
    checkpoint.restore(payload, hypernet=hypernet, uncle=trainer)
    del payload
    return target, hypernet, trainer, config, path


def starting_code(config, task):
    """The code this group began with, before a single update touched it."""
    return torch.randn(
        config.code_dim, device=config.torch_device,
        generator=random_stream(config.seed, "code", task, config.torch_device))

## 6. The three arms

`fresh_buffers` is the part worth reading slowly. BatchNorm keeps a running
mean and variance for every channel. Training mode is what makes it write those
statistics back, and the tensors passed in are the ones it writes to, so
starting from the neutral template and running a few batches gives one arm its
own set. No gradients are taken and no parameter is updated: this is a
measurement that happens to need a forward pass.

In [ ]:
def fresh_buffers(target, trainer, task, weights, *, batches=BUFFER_BATCHES):
    """Running statistics that belong to these weights and no others."""
    buffers = {name: value.clone() for name, value in trainer.buffer_template.items()}
    loader = DataLoader(tasks[task]["train"], batch_size=BATCH_SIZE, shuffle=False)
    was_training = target.training
    target.train()
    try:
        with torch.no_grad():
            for index, batch in enumerate(loader):
                if index >= batches:
                    break
                functional_call(target, (weights, buffers),
                                (batch[0].to(trainer.device),), strict=True)
    finally:
        target.train(was_training)
    return buffers


def arms_for_model(order, seed, point):
    """Three codes per learned group, scored on that group's own test pictures."""
    target, hypernet, trainer, config, path = build(order, seed, point)
    try:
        learned = list(hypernet.task_codes)
        # Where each group sits in this stream, counting learn requests only.
        position = {name: index for index, (kind, name) in enumerate(
            [request for request in config.requests if request[0] == "learn"], start=1)}
        rows = []
        for task in learned:
            trained = hypernet.task_codes[task].detach()
            # The next learned group's code, wrapping round. Any other group's
            # code would do. This one is picked by a rule so that the choice
            # cannot drift between runs.
            other_task = learned[(learned.index(task) + 1) % len(learned)]
            codes = {
                "trained": trained,
                "initial": starting_code(config, task),
                "other": hypernet.task_codes[other_task].detach(),
            }
            for arm, code in codes.items():
                weights = hypernet.weights_from_code(code)
                scored = probe(hypernet, target, trainer, tasks[task]["test"],
                               code_task=task, classes=CLASSES, weights=weights,
                               buffers=fresh_buffers(target, trainer, task, weights))
                rows.append(dict(
                    order=order, seed=seed, point=point, task=task,
                    position=position.get(task), arm=arm,
                    other_task=other_task if arm == "other" else None,
                    statistics="fresh", accuracy=scored["accuracy"],
                    distinct_classes=scored["distinct_classes"],
                    input_sensitivity=scored["input_sensitivity"],
                    code_revision=HEAD, device=GPU_NAME))
            # The check described at the top: the trained arm again, through the
            # statistics the run itself saved. If this disagrees with the fresh
            # reading, the re-estimation is doing something of its own and the
            # three arms above cannot be compared.
            scored = probe(hypernet, target, trainer, tasks[task]["test"],
                           code_task=task, classes=CLASSES)
            rows.append(dict(
                order=order, seed=seed, point=point, task=task,
                position=position.get(task), arm="trained", other_task=None,
                statistics="as_run", accuracy=scored["accuracy"],
                distinct_classes=scored["distinct_classes"],
                input_sensitivity=scored["input_sensitivity"],
                code_revision=HEAD, device=GPU_NAME))
        return rows
    finally:
        del trainer, hypernet, target
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        # One model file at a time, so several runs do not fill the disk.
        if path is not None and Path(path).exists():
            Path(path).unlink()

## 7. Run it

Every finished model is uploaded before the next one starts, so stopping
halfway costs nothing and starting again skips what is already there.

In [ ]:
rows = []
if store.info(RESULT_KEY) is not None:
    text = store.download(RESULT_KEY, CACHE).read_text(encoding="utf-8")
    rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    print(f"Resuming: {len(rows)} readings already saved.")

PARTIAL = CACHE / "arms_partial.jsonl"
PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in rows), encoding="utf-8")

began = time.perf_counter()
for order, seed in tqdm(RUNS, desc="runs", unit="run"):
    for point in POINTS:
        if any((order, seed, point) == (r["order"], r["seed"], r["point"]) for r in rows):
            continue
        new = arms_for_model(order, seed, point)
        rows += new
        with PARTIAL.open("a", encoding="utf-8") as handle:
            for row in new:
                handle.write(json.dumps(row) + "\n")
        store.upload(PARTIAL, RESULT_KEY, overwrite=True)
print(f"\n{len(rows)} readings in {(time.perf_counter() - began) / 60:.1f} minutes.")

## 8. What it found

Three things, in order. The check first: the trained arm read two ways. Then
the three arms, averaged over seeds. Then the verdict, which is a rule written
down before the numbers arrived, so that reading them cannot change it.

In [ ]:
import statistics
from collections import defaultdict


def mean(values):
    values = list(values)
    return sum(values) / len(values) if values else float("nan")


def spread(values):
    values = list(values)
    return statistics.stdev(values) if len(values) > 1 else 0.0


# ---- the check -----------------------------------------------------------
print("CHECK: the trained arm, through fresh statistics and through the saved ones")
fresh = {(r["order"], r["seed"], r["point"], r["task"]): r["accuracy"]
         for r in rows if r["arm"] == "trained" and r["statistics"] == "fresh"}
as_run = {(r["order"], r["seed"], r["point"], r["task"]): r["accuracy"]
          for r in rows if r["arm"] == "trained" and r["statistics"] == "as_run"}
gaps = [abs(fresh[key] - as_run[key]) for key in fresh if key in as_run]
print(f"  {len(gaps)} pairs | largest gap {max(gaps):.2f} points "
      f"| mean gap {mean(gaps):.2f} points")
if gaps and max(gaps) > 2.0:
    print("  WARNING: re-estimating the statistics moved the trained arm by more\n"
          "  than 2 points. The arms below are not comparable. Stop here.")
else:
    print("  The re-estimation is harmless, so the three arms differ only by the code.")

# ---- the three arms ------------------------------------------------------
print(f"\nTHE THREE ARMS, mean over seeds, percent correct, chance is {CHANCE:.1f}")
print(f"{'group':>7}{'position':>10}{'trained':>10}{'initial':>10}{'other':>10}")
by_arm = defaultdict(list)
for r in rows:
    if r["statistics"] == "fresh":
        by_arm[(r["task"], r["arm"])].append(r["accuracy"])
groups = sorted({r["task"] for r in rows}, key=int)
positions = {r["task"]: r["position"] for r in rows}
for task in groups:
    print(f"{task:>7}{str(positions[task]):>10}"
          f"{mean(by_arm[(task, 'trained')]):>10.2f}"
          f"{mean(by_arm[(task, 'initial')]):>10.2f}"
          f"{mean(by_arm[(task, 'other')]):>10.2f}")

overall = {arm: mean(value for (task, kind), values in by_arm.items() if kind == arm
                     for value in values)
           for arm in ("trained", "initial", "other")}
seed_spread = mean(spread(by_arm[(task, "trained")]) for task in groups)
print(f"{'all':>7}{'':>10}{overall['trained']:>10.2f}{overall['initial']:>10.2f}"
      f"{overall['other']:>10.2f}")
print(f"\nSpread between seeds on the trained arm: {seed_spread:.2f} points. "
      f"That is the size of a difference that means nothing.")

# ---- the verdict ---------------------------------------------------------
# An arm counts as near chance if it sits within two seed spreads of chance,
# and near trained if it sits within two seed spreads of the trained arm. Two
# spreads rather than one, because the claim either way is a strong one. The
# floor of 1 point stops a run with a tiny spread from calling everything a
# difference.
margin = max(2 * seed_spread, 1.0)
near_chance = overall["initial"] - CHANCE <= margin, overall["other"] - CHANCE <= margin
near_trained = overall["trained"] - overall["initial"] <= margin

print("\nVERDICT")
if all(near_chance):
    print("  The generator conditions on the code. A group's own trained code is\n"
          "  what makes its network work, and no other code stands in for it.\n"
          "  The question closes.")
elif near_trained and near_chance[1]:
    print("  The generator conditions on something coarse: enough to tell groups\n"
          "  apart, since another group's code fails, but not enough to need the\n"
          "  trained code, since the starting random one works.")
elif near_trained:
    print("  The generator does NOT need the code for a group it has learned. The\n"
          "  starting random code scores as well as the trained one, so for these\n"
          "  groups the 32 numbers are decoration. This belongs in the paper.")
else:
    print("  In between. The code carries some of a group's accuracy but not all\n"
          f"  of it: trained {overall['trained']:.2f}, initial {overall['initial']:.2f}, "
          f"other {overall['other']:.2f},\n  chance {CHANCE:.1f}. Report the three "
          "numbers and do not reduce them to a yes or no.")